 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [33]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
citenum_url_link_re = re.compile(r'\[(?P<orig>\d+)\]\((?P<url>https?://[^\)]+)\)')
perplex_source_list_re = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)')
citenum_plain_re = re.compile(r'\[(?P<num>\d+)\]')

def dedup_citenums_to_urls(num_url_pairs: list[tuple[str, str]], verbose: bool = False) -> pd.DataFrame:
    """Return a dataframe showing remapping when >1 citenums map to the same URL."""
    url_to_citenums = defaultdict(list)
    for num, url in num_url_pairs:
        url_to_citenums[url].append(num)
    
    # Create new citation numbers if there are duplicates
    new_cite_num = 1
    lut = []
    found_dups = False
    for url, nums in url_to_citenums.items():
        if (nDups := len(nums)) > 1 and verbose:
            found_dups = True
            print(f'URL has {nDups} dups: {nums=}, {url=}')
            
        for num in nums:
            lut.append({'orig_num': num, 'new_num': str(new_cite_num), 'url': url})
        
        new_cite_num += 1
    
    citenums_to_url = pd.DataFrame(lut).set_index(['orig_num'])
    if verbose and found_dups:
        display(citenums_to_url)
    
    return citenums_to_url

def replace_body_plain_citenum(body: str, oldnum_to_new: Dict[str, str]) -> str:
    """Replace plain citation numbers in the body with new ones."""

    def replace(match):
        return f'[{oldnum_to_new[match.group("num")]}]'

    return re.sub(citenum_plain_re, replace, body)

def collect_and_fix_body_links(file_text: str, verbose: bool = False) -> tuple[str, pd.DataFrame]:
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes. """    

    section_parts = file_text.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts

    # Reassign body cite numbers if duplicate URLs are found in the sources
    source_matches = list(perplex_source_list_re.finditer(citations))
    num_url_pairs = [(match.group('num'), match.group('url')) for match in source_matches]
    citenums_to_url = dedup_citenums_to_urls(num_url_pairs, verbose=verbose)
    
    body = replace_body_plain_citenum(body, citenums_to_url.new_num.to_dict())
    
    return body, citenums_to_url

def relink_chunks(body: str, citenums_to_url: pd.DataFrame) -> tuple[str, str]:
    """Replaces body links with Zotero or Obsidian links, and returns the relinked body and sources."""
    def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
        """Returns what a relinked citation would look like if present in the body,
        given a source citation number and url.  Also appends a relinked source to
        relinked_sources, and expects the set "body_cite_nums"."""
        
        numbered_link = f"[{cite_num}]({doc_url})"
        if zotero_item := relinker.find_zotero_item_via_url(doc_url):
            body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
            relinked_sources.append(f'({numbered_link}) **{body_link}**')
        else:
            body_link = f"=={numbered_link}==" # mark it as "not in zotero"
            source_line = f'({numbered_link}) {doc_url}'
            source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
            relinked_sources.append(source_line)
            
        return body_link

    # get the map from citenums, in case they had to be redone    
    new_num_to_url = citenums_to_url.set_index('new_num').url.to_dict()
    
    # globals for make_relinks_from_source()
    body_cite_nums = set(re.findall(citenum_plain_re, body))
    relinked_sources = []
    relinker = lpz.ZoteroLinkConverter()

    # compute links to zotero and obsidian, when possible
    source_num_to_link = {num: make_relinks_from_source(num, url)
                          for num, url in new_num_to_url.items() }
    # replace the cite numbers with new links
    body_relinked = re.sub(citenum_plain_re, 
                           lambda m: f' {source_num_to_link.get(m.group("num"))}', body)
    # relinked sources were stored in this global
    sources_relinked = "\n".join(relinked_sources)
    
    return body_relinked, sources_relinked

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    file_text = perplexity_file.read_text(encoding='utf-8')
    body, citenums_to_url = collect_and_fix_body_links(file_text, verbose=verbose)
    body_relinked, sources_relinked = relink_chunks(body, citenums_to_url)
    relinked_file.write_text(f'# Response\n{body_relinked}\n# Citations\n{sources_relinked}', encoding='utf-8')

In [35]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = True
relink_perplexity_export(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
URL has 2 dups: nums=['4', '54'], url='https://globalaffairs.org/commentary-and-analysis/blogs/brazils-systemic-mistrust-elections-and-democracy'


,new_num,url
orig_num,,
1,1,https://en.wikipedia.org/wiki/Right-wing_populism
2,2,https://www.politico.eu/article/mapped-europe-...
3,3,https://www.npr.org/2024/06/09/nx-s1-4997712/f...
4,4,https://globalaffairs.org/commentary-and-analy...
54,4,https://globalaffairs.org/commentary-and-analy...
...,...,...
77,76,https://rioonwatch.org/?p=72542
78,77,https://www.populismstudies.org/chega-emerges-...
79,78,https://www.american.edu/sis/centers/transatla...


Reading from cache.
Done.


### Test merging

In [4]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

chat_files = list(datdir.glob('*.md'))
chat_files

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

##### Fix up the cite numbers within each body and collect them

In [5]:
verbose = False
all_bodies, all_citenums_to_url = [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'{chat_file.stem}')
    file_text = chat_file.read_text(encoding='utf-8')
    body, citenums_to_url = collect_and_fix_body_links(file_text, verbose=verbose)
    all_bodies.append(body)
    citenums_to_url['file_index']=file_index
    citenums_to_url['chat_file']=chat_file
    all_citenums_to_url.append(citenums_to_url.reset_index())
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} citation numbers')

In [6]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['new_num_int'] = df['new_num'].astype(int)

grouped = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_new_num_int=('new_num_int', 'mean')
).reset_index()

grouped = grouped.sort_values(by=['mean_file_index', 'mean_new_num_int'], 
                              ascending=True).reset_index(drop=True)

grouped['citenum_merged'] = np.arange(1, len(grouped) + 1) # citenum == rank

# Merge back the new citenumes
df = df.merge(grouped[['url', 'citenum_merged']], on='url')

all_citenums_to_url = df.sort_values('citenum_merged')

 # NEXT
- go back to the in individual fixed-up docs are reorder each body using this table
- then merge them all together
- make the new sources.  Somehow have to rebuild it, keeping track of which entries in the final sources doc and zotero/obsidian notes, and if they were actually cited.
  - might have to strip out the non-zotero, non-obs cites from the merged body

#### Assign unified cite numbers to each body and then merge

In [ ]:
# new_num_to_url = citenums_to_url.set_index('new_num').url.to_dict()


,new_num,url,file_index,chat_file
orig_num,,,,
1,1,https://www.bbc.com/news/world-europe-36130006,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
2,2,https://www.politico.eu/article/mapped-europe-...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
3,3,https://forum.lasaweb.org/files/vol54-issue4/d...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
4,4,https://www.euronews.com/my-europe/2024/04/23/...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
5,5,https://www.bbc.com/news/world-europe-63029909,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
6,6,https://dgap.org/en/research/publications/righ...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
7,7,https://www.pewresearch.org/global/2024/12/11/...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
8,8,https://www.pewresearch.org/short-reads/2022/1...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...
9,9,https://foreignpolicy.com/2023/12/26/right-win...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...


In [ ]:
all_citenums_to_url[['citenum_merged','url']].rename()

,orig_num,new_num,url,file_index,chat_file,new_num_int,citenum_merged
43,4,4,https://www.euronews.com/my-europe/2024/04/23/...,2,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,4,1
23,4,4,https://www.euronews.com/my-europe/2024/04/23/...,1,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,4,1
313,4,4,https://www.euronews.com/my-europe/2024/04/23/...,7,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,4,1
83,4,4,https://www.euronews.com/my-europe/2024/04/23/...,4,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,4,1
63,4,4,https://www.euronews.com/my-europe/2024/04/23/...,3,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,4,1
...,...,...,...,...,...,...,...
304,77,77,https://blogs.lse.ac.uk/europpblog/2020/07/30/...,6,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,77,189
305,78,78,https://cris.maastrichtuniversity.nl/ws/portal...,6,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,78,190
307,80,80,https://www.osw.waw.pl/en/publikacje/osw-comme...,6,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,80,191
308,81,81,https://pmc.ncbi.nlm.nih.gov/articles/PMC10081...,6,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,81,192


In [21]:
df = all_citenums_to_url.set_index('file_index')[['new_num','citenum_merged','url']]

file_index=0
df_this = df.loc[file_index]
df_this

,new_num,citenum_merged,url
file_index,,,
0,4,1,https://www.euronews.com/my-europe/2024/04/23/...
0,5,2,https://www.bbc.com/news/world-europe-63029909
0,8,3,https://www.pewresearch.org/short-reads/2022/1...
0,14,4,https://whogoverns.eu/in-polarised-spain-popul...
0,17,5,https://theloop.ecpr.eu/how-real-is-the-threat...
0,19,6,https://foreignpolicy.com/2024/06/28/far-right...
0,6,7,https://dgap.org/en/research/publications/righ...
0,11,8,https://carnegieendowment.org/posts/2022/09/ho...
0,12,9,https://dcubrexitinstitute.eu/2023/10/election...


In [27]:
citenum_current_to_merged = df_this.set_index('new_num').citenum_merged.to_dict()
citenum_current_to_merged

{'4': 1,
 '5': 2,
 '8': 3,
 '14': 4,
 '17': 5,
 '19': 6,
 '6': 7,
 '11': 8,
 '12': 9,
 '2': 10,
 '16': 11,
 '18': 12,
 '7': 13,
 '9': 14,
 '3': 15,
 '13': 16,
 '15': 17,
 '1': 18,
 '20': 19,
 '10': 20}

In [29]:
body = all_bodies[file_index]
print(body)

Two notable defeats of right-wing populist governments have occurred in recent years:

## Poland
A broad coalition led by Donald Tusk defeated the ruling Law and Justice Party (PiS) in late 2023[6][12]. The victory came after PiS had held power for nearly a decade and had quadrupled their vote share between 2001-2019[8]. The defeat was achieved through:

**Coalition Building**
- Three opposition parties formed a broad coalition of diverse political actors[12]
- The unified opposition strategy proved effective against the incumbent PiS

## Finland
The Finns Party suffered a significant electoral defeat in the 2024 European Parliament elections:
- Dropped to sixth place nationally
- Lost 6.2% of their previous vote share[18]

## Attempted but Failed Defeats

**Spain**
Despite losing over 600,000 votes in the most recent general election, the far-right maintains influence due to high political polarization[4][14]. The country remains in a paradoxical situation where populist forces retain

In [7]:
concat_bodies = ""
concat_sources = ""
for file_index, body in enumerate(all_bodies):
    # relink body with old citenums
    source_matches = citenums_to_url[file_index]
    url_to_source_nums = all_url_to_source_nums[file_index]
    body_relinked, sources_relinked = relink_chunks(body, source_matches, citenums_to_url)
    
    citenums_to_url =lut.loc[file_index].new_cite_num.to_dict()
    body_re_relinked = re.sub(citenum_url_link_re, replace_link_num, body_relinked)
    concat_bodies += f'# {chat_files[file_index].name}\n{body_re_relinked}\n'
    sources_re_relinked = re.sub(citenum_url_link_re, replace_link_num, sources_relinked)
    concat_sources += f'{sources_re_relinked}\n'

KeyError: 0

In [ ]:
import numpy as np
len(concat_sources.split('\n')), len(np.unique(concat_sources.split('\n')))

concat_sources_unique = list(set(concat_sources.split('\n')))
#sorted_strings = sorted(concat_sources_unique, key=lambda x: int(re.search(r'\((\d+)\]', x).group(1)))
#sorted_strings

# Function to extract the number inside [num]
def extract_number(s):
    match = re.search(r'\[(\d+)\]', s)
    return int(match.group(1)) if match else None  # Handle cases without [num]

# Sort the list using the extracted number as key
merged_sources = "\n".join(sorted(concat_sources_unique, key=extract_number))

ic(merged_output_file)
merged_output_file.write_text(f'# Responses\n{concat_bodies}\n# Citations\n{merged_sources}', encoding='utf-8')
print('Done.')